In [1]:
import pandas as pd 
import numpy as np 

X_tr = pd.read_csv('../data/X_tr.csv')
X_test = pd.read_csv('../data/X_test.csv')
y = pd.read_csv('../data/y.csv')

y = np.ravel(y)

In [2]:
import torch_directml 
device = torch_directml.device()
device

device(type='privateuseone', index=0)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_tr, y, test_size=0.25, random_state=42)

In [9]:
X_train.shape, y_train.shape

((472500, 16), (472500,), dtype('int64'))

In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import numpy as np


class NN(nn.Module):
    def __init__(self, input_dim):
        super(NN, self).__init__()
        self.lin_1 = nn.Linear(input_dim, 64)
        self.lin_2 = nn.Linear(64, 32)
        self.lin_3 = nn.Linear(32, 16)
        self.lin_4 = nn.Linear(16, 1)
        self.bn_1 = nn.BatchNorm1d(64)
        self.bn_2 = nn.BatchNorm1d(32)
        self.bn_3 = nn.BatchNorm1d(16)
        self.func = nn.ReLU()
        self.dropout = nn.Dropout(0.3) 
     
    def forward(self, x):
        x = self.func(self.bn_1(self.lin_1(x)))
        x = self.dropout(x)
        x = self.func(self.bn_2(self.lin_2(x)))
        x = self.dropout(x)
        x = self.func(self.bn_3(self.lin_3(x)))
        x = self.dropout(x)
        x = self.lin_4(x)
        
        return x

def to_torch(X, y=None):
    X_tensor = torch.FloatTensor(X.values)

    if y is not None:
        y_tensor = torch.FloatTensor(y).view(-1, 1)
        
        return TensorDataset(X_tensor, y_tensor)
    
    return X_tensor

train_dataset = to_torch(X_train, y_train)
val_dataset = to_torch(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size = 2048, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 2048, shuffle=False)


model = NN(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

In [47]:
X_test.describe()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
count,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000,270000.000000
mean,0.002806,0.003499,0.003850,0.000993,0.000238,-0.001727,0.002086,0.002166,0.006391,0.004709,0.000462,0.118785,0.311267,0.524596,0.002167,0.488163
std,0.999900,0.998330,1.002145,1.000413,1.000370,1.002121,1.001058,0.998889,1.001825,1.003408,1.000137,0.323536,0.463012,0.499396,0.046497,0.499861
min,-3.044551,-1.582881,-2.437096,-3.533442,-0.294858,-4.280706,-0.613913,-0.754928,-0.836168,-0.564825,-0.830189,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-0.743276,-1.582881,-0.700960,-0.653527,-0.294858,-0.565940,-0.613913,-0.754928,-0.836168,-0.564825,-0.830189,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-0.016558,0.631760,-0.033216,-0.059730,-0.294858,0.218870,-0.613913,-0.649495,-0.836168,-0.564825,-0.830189,0.000000,0.000000,1.000000,0.000000,0.000000
75%,0.710160,0.631760,0.634529,0.712205,-0.294858,0.689756,1.628894,0.721132,0.998051,0.687448,1.221087,0.000000,1.000000,1.000000,0.000000,1.000000
max,2.769196,0.631760,4.640995,9.470709,3.391458,2.573299,1.628894,5.781909,2.832269,3.191993,1.221087,1.000000,1.000000,1.000000,1.000000,1.000000


In [48]:
epochs = 50
best_auc = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
  
    model.eval()
    val_preds = []
    val_true = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            val_preds.append(outputs.cpu().numpy())
            val_true.append(y_batch.numpy())
    
    val_preds = np.concatenate(val_preds).flatten()
    val_true = np.concatenate(val_true).flatten()
    
    
    val_auc = roc_auc_score(val_true, val_preds)
    val_acc = accuracy_score(val_true, (val_preds > 0.5).astype(int))
    
    #scheduler.step(val_auc)
  
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'best_nn_model.pth')
  
    
    if epoch % 10 == 0:
        print(f"epoch {epoch}/{epochs}, loss: {train_loss/len(train_loader)}, "
              f"acc: {val_acc}, roc-auc: {val_auc}")

epoch 0/50, loss: 0.3802862870486784, acc: 0.8825015873015873, roc-auc: 0.9517355300034538
epoch 10/50, loss: 0.28811646449617495, acc: 0.8813714285714286, roc-auc: 0.9527406415612083
epoch 20/50, loss: 0.2876308219128357, acc: 0.8807492063492064, roc-auc: 0.9527834001085158
epoch 30/50, loss: 0.2873881606312541, acc: 0.8821460317460318, roc-auc: 0.9528022747939071
epoch 40/50, loss: 0.2872973893369947, acc: 0.8813714285714286, roc-auc: 0.952855970078615


In [49]:
subm = pd.read_csv('../data/sample_submission.csv')

model = NN(X_train.shape[1])  
model.load_state_dict(torch.load('best_nn_model.pth'))  
model.to(device)  
X_test_tor = to_torch(X_test).to(device)

model.eval()
with torch.no_grad():
    pr = model(X_test_tor).cpu().numpy().flatten()


subm['Heart Disease'] = pr
subm.to_csv('../data/subm/nn_3.0.csv', index=False)

C:\Users\Professional\AppData\Local\Temp\ipykernel_16832\2580389773.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_nn_model.pth')

модель проиграла обычным бустингам в качестве
 

adacap-пока отмена(на маленьких датасетах полезна)

In [53]:
class AdaCapNet(nn.Module):
    def __init__(self, input_dim=16, hidden_dim=32):
        super().__init__()
        # Простой энкодер
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # Обучаемый параметр регуляризации
        self.log_lambda = nn.Parameter(torch.tensor(0.0))
        
    def forward(self, x, y=None):
        features = self.encoder(x)
        
        if y is None or not self.training:
            # На инференсе используем простой линейный слой
            if not hasattr(self, 'classifier'):
                self.classifier = nn.Linear(features.shape[1], 1).to(x.device)
            return self.classifier(features)
        
        # Тихоновское решение во время обучения
        lam = torch.exp(self.log_lambda)
        
        # Решаем (H^T H + λI) W = H^T y
        H = features
        y = y.float().unsqueeze(1)
        
        HtH = H.T @ H
        HtY = H.T @ y
        
        I = torch.eye(H.shape[1], device=H.device)
        W = torch.linalg.solve(HtH + lam * I, HtY)
        
        return H @ W


def adacap_loss(model, x, y, num_perm=5):
    """Минимальная реализация AdaCap loss"""
    # Ошибка на реальных данных
    pred_real = model(x, y)
    loss_real = nn.functional.binary_cross_entropy_with_logits(
        pred_real.squeeze(), y.float()
    )
    
    # Ошибка на перемешанных данных
    loss_perm = 0
    for _ in range(num_perm):
        y_perm = y[torch.randperm(len(y))]
        pred_perm = model(x, y_perm)
        loss_perm += nn.functional.binary_cross_entropy_with_logits(
            pred_perm.squeeze(), y_perm.float()
        )
    loss_perm /= num_perm
    
    # Контрастивная потеря: реальная ошибка МИНУС ошибка на перемешанных
    return loss_real - 0.5 * loss_perm
def train_adacap(model, X_train, y_train, X_val, y_val, 
                 epochs=200, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train)
    X_val_t = torch.FloatTensor(X_val)
    y_val_t = torch.FloatTensor(y_val)
    
    best_acc = 0
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        loss = adacap_loss(model, X_train_t, y_train_t)
        loss.backward()
        optimizer.step()
        
        # Валидация
        if epoch % 20 == 0:
            model.eval()
            with torch.no_grad():
                pred_val = torch.sigmoid(model(X_val_t)).squeeze()
                acc = ((pred_val > 0.5) == y_val_t).float().mean().item()
                
                if acc > best_acc:
                    best_acc = acc
                    
            print(f'Epoch {epoch}, loss: {loss.item():.4f}, val_acc: {acc:.4f}')
    
    return best_acc


model = AdaCapNet(input_dim=16, hidden_dim=32)
best_acc = train_adacap(model, X_train.values, y_train, X_val.values, y_val)

print(f"\nЛучшая точность на валидации: {best_acc:.4f}")

Epoch 0, loss: 0.2616, val_acc: 0.4479
Epoch 20, loss: 0.2354, val_acc: 0.4478
Epoch 40, loss: 0.2294, val_acc: 0.4478
Epoch 60, loss: 0.2249, val_acc: 0.4476
Epoch 80, loss: 0.2184, val_acc: 0.4468
Epoch 100, loss: 0.2085, val_acc: 0.4478
Epoch 120, loss: 0.2020, val_acc: 0.4479


KeyboardInterrupt: 